# 02. 탑승 수요 분석

실제 탑승 기준 시간대별, 요일별, 요일 X 시간대별 패턴과 일평균 보정 결과를 생성한다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from matplotlib.colors import Normalize
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

with open('seoul_municipalities_geo_simple.json', encoding='utf-8') as map_file:
    seoul_map = json.load(map_file)

In [ ]:
df2 = pd.read_csv('data/서울시설공단_장애인콜택시 시간대별 탑승건수_20251231.csv')
print(df2.info())
print(df2.shape)

In [ ]:
df2.head()

## 시간대별 탑승 히트맵

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

numeric_columns = df2.select_dtypes(include='number').columns
demand_column = numeric_columns[0]
heatmap_data = (
    df2.assign(시간=pd.to_datetime(df2['시간대'], format='%H:%M:%S').dt.hour)
    .groupby('시간')[demand_column]
    .sum()
    .sort_values(ascending=False)
    .to_frame()
)
heatmap_data.index = [f'{hour:02d}시' for hour in heatmap_data.index]

fig, (ax_heatmap, ax_bar) = plt.subplots(
    1, 2, figsize=(16, 10), gridspec_kw={'width_ratios': [1, 1.8]}
)
sns.heatmap(
    heatmap_data,
    ax=ax_heatmap,
    cmap='YlOrRd',
    annot=True,
    fmt=',.0f',
    linewidths=0.5,
    cbar=False,
)
ax_heatmap.set_title('시간대 탑승 히트맵')
ax_heatmap.set_xlabel('탑승 지표')
ax_heatmap.set_ylabel('시간대')
ax_heatmap.tick_params(axis='y', rotation=0)

ax_bar.barh(heatmap_data.index, heatmap_data[demand_column], color='#e85d04')
ax_bar.invert_yaxis()
ax_bar.set_title('시간대별 탑승 순위')
ax_bar.set_xlabel('탑승건수')
ax_bar.set_ylabel('')
for index, value in enumerate(heatmap_data[demand_column]):
    ax_bar.text(value, index, f' {value:,.0f}', va='center')

plt.tight_layout()
plt.show()

## 요일별 탑승 히트맵

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

weekday_order = ['월요일', '화요일', '수요일', '목요일', '금요일', '토요일', '일요일']
weekday_summary = (
    df2.assign(
        요일=pd.to_datetime(df2['일자']).dt.dayofweek.map(dict(enumerate(weekday_order)))
    )
    .groupby('요일')[demand_column]
    .sum()
    .reindex(weekday_order)
    .to_frame(name='탑승건수')
)

fig, (ax_heatmap, ax_bar) = plt.subplots(
    1, 2, figsize=(16, 8), gridspec_kw={'width_ratios': [1, 1.8]}
)
sns.heatmap(
    weekday_summary,
    ax=ax_heatmap,
    cmap='YlOrRd',
    annot=True,
    fmt=',.0f',
    linewidths=0.5,
    cbar=False,
)
ax_heatmap.set_title('요일별 탑승 히트맵')
ax_heatmap.set_xlabel('탑승 지표')
ax_heatmap.set_ylabel('요일')
ax_heatmap.tick_params(axis='y', rotation=0)

weekday_ranking = weekday_summary['탑승건수'].sort_values(ascending=True)
ax_bar.barh(weekday_ranking.index, weekday_ranking.values, color='#e85d04')
ax_bar.invert_yaxis()
ax_bar.set_title('요일별 탑승 순위')
ax_bar.set_xlabel('탑승건수')
ax_bar.set_ylabel('')
for index, value in enumerate(weekday_ranking.values):
    ax_bar.text(value, index, f' {value:,.0f}', va='center')

plt.tight_layout()
plt.show()

## 요일별 X 시간대별 탑승 히트맵

In [ ]:
weekday_order = ['월요일', '화요일', '수요일', '목요일', '금요일', '토요일', '일요일']
weekday_data = df2.assign(
    요일=pd.to_datetime(df2['일자']).dt.dayofweek.map(dict(enumerate(weekday_order))),
    시간=pd.to_datetime(df2['시간대'], format='%H:%M:%S').dt.hour,
)

weekday_heatmap = weekday_data.pivot_table(
    index='요일', columns='시간', values='탑승건수', aggfunc='sum'
).reindex(weekday_order)
weekday_heatmap.columns = [f'{hour:02d}시' for hour in weekday_heatmap.columns]
weekday_ranking = weekday_heatmap.sum(axis=1).sort_values(ascending=False)

fig, (ax_heatmap, ax_bar) = plt.subplots(
    2, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [3.5, 1.4]}
)
sns.heatmap(
    weekday_heatmap,
    ax=ax_heatmap,
    cmap='YlOrRd',
    annot=True,
    fmt=',.0f',
    linewidths=0.5,
    cbar_kws={'label': '탑승건수'},
)
ax_heatmap.set_title('요일별 시간대 탑승 히트맵', pad=12)
ax_heatmap.set_xlabel('시간대')
ax_heatmap.set_ylabel('요일')
ax_heatmap.tick_params(axis='x', rotation=0)
ax_heatmap.tick_params(axis='y', rotation=0)

ax_bar.barh(weekday_ranking.index, weekday_ranking.values, color='#e85d04')
ax_bar.invert_yaxis()
ax_bar.set_title('요일별 탑승 순위', pad=12)
ax_bar.set_xlabel('총 탑승건수')
ax_bar.set_ylabel('')
for index, value in enumerate(weekday_ranking.values):
    ax_bar.text(value, index, f' {value:,.0f}', va='center')

plt.tight_layout()
plt.show()

## 요일 × 시간대별 탑승 비율 히트맵

In [ ]:
weekday_ratio = weekday_heatmap.div(weekday_heatmap.sum(axis=1), axis=0) * 100
weekday_total = weekday_heatmap.sum(axis=1)
weekday_share = (weekday_total / weekday_total.sum() * 100).sort_values(ascending=False)

fig, (ax_heatmap, ax_bar) = plt.subplots(
    2, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [3.5, 1.4]}
)
sns.heatmap(
    weekday_ratio,
    ax=ax_heatmap,
    cmap='YlGnBu',
    annot=True,
    fmt='.1f',
    linewidths=0.5,
    cbar_kws={'label': '요일 내 시간대 비율 (%)'},
)
ax_heatmap.set_title('요일 × 시간대별 탑승 비율 히트맵', pad=12)
ax_heatmap.set_xlabel('시간대')
ax_heatmap.set_ylabel('요일')
ax_heatmap.tick_params(axis='x', rotation=0)
ax_heatmap.tick_params(axis='y', rotation=0)

ax_bar.bar(weekday_share.index, weekday_share.values, color='#0077b6')
ax_bar.set_title('전체 탑승 중 요일별 비중', pad=12)
ax_bar.set_xlabel('요일')
ax_bar.set_ylabel('전체 탑승 비중 (%)')
ax_bar.tick_params(axis='x', rotation=0)
for index, value in enumerate(weekday_share.values):
    ax_bar.text(index, value, f'{value:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 탑승 집계 CSV 생성


In [ ]:
# 1. 공통 준비

import pandas as pd
from pathlib import Path

input_path = Path('data/서울시설공단_장애인콜택시 시간대별 탑승건수_20251231.csv')
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

weekday_order = ['월요일', '화요일', '수요일', '목요일', '금요일', '토요일', '일요일']
weekday_map = dict(enumerate(weekday_order))

df_boarding_hour = pd.read_csv(input_path)

df_boarding_hour['일자'] = pd.to_datetime(df_boarding_hour['일자'], errors='coerce')
df_boarding_hour['요일'] = df_boarding_hour['일자'].dt.dayofweek.map(weekday_map)
df_boarding_hour['시간'] = pd.to_datetime(
    df_boarding_hour['시간대'],
    format='%H:%M:%S',
    errors='coerce'
).dt.hour
df_boarding_hour['시간대_label'] = df_boarding_hour['시간'].map(
    lambda x: f'{int(x):02d}시' if pd.notna(x) else pd.NA
)

total_service_days = df_boarding_hour['일자'].dropna().dt.date.nunique()

weekday_service_days = (
    df_boarding_hour[['일자', '요일']]
    .dropna()
    .drop_duplicates()
    .groupby('요일')
    .size()
    .reindex(weekday_order)
    .rename('운행일수')
)

hour_columns = [f'{hour:02d}시' for hour in range(24)]

print('공통 준비 완료')
print(f'전체 운행일수: {total_service_days}')
print(weekday_service_days)

In [ ]:
# 2. 요일별_전체_탑승건수.csv 생성

weekday_total = (
    df_boarding_hour.groupby('요일', as_index=False)['탑승건수']
    .sum()
)

weekday_total['요일'] = pd.Categorical(
    weekday_total['요일'],
    categories=weekday_order,
    ordered=True
)

weekday_total = weekday_total.sort_values('요일').reset_index(drop=True)

output_path = output_dir / '요일별_전체_탑승건수.csv'
weekday_total.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'저장 완료: {output_path}')
weekday_total

In [ ]:
# 3. 시간대별_전체_탑승건수.csv 생성

hour_total = (
    df_boarding_hour.groupby(['시간', '시간대_label'], as_index=False)['탑승건수']
    .sum()
    .sort_values('시간')
    .reset_index(drop=True)
)

hour_total_csv = (
    hour_total[['시간대_label', '탑승건수']]
    .rename(columns={'시간대_label': '시간대'})
)

output_path = output_dir / '시간대별_전체_탑승건수.csv'
hour_total_csv.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'저장 완료: {output_path}')
hour_total_csv

In [ ]:
# 4. 요일X시간대별_전체_탑승건수.csv 생성

weekday_hour_total = (
    df_boarding_hour.pivot_table(
        index='요일',
        columns='시간대_label',
        values='탑승건수',
        aggfunc='sum',
        fill_value=0
    )
    .reindex(weekday_order)
)

weekday_hour_total = weekday_hour_total[
    [col for col in hour_columns if col in weekday_hour_total.columns]
]

weekday_hour_total.index.name = '요일'

output_path = output_dir / '요일X시간대별_전체_탑승건수.csv'
weekday_hour_total.to_csv(output_path, encoding='utf-8-sig')

print(f'저장 완료: {output_path}')
weekday_hour_total

In [ ]:
# 5. 요일별_일평균_탑승건수.csv 생성

weekday_total_series = (
    df_boarding_hour.groupby('요일')['탑승건수']
    .sum()
    .reindex(weekday_order)
    .rename('총탑승건수')
)

weekday_daily_avg = pd.concat(
    [weekday_service_days, weekday_total_series],
    axis=1
)

weekday_daily_avg['일평균탑승건수'] = (
    weekday_daily_avg['총탑승건수'] / weekday_daily_avg['운행일수']
)

weekday_daily_avg = (
    weekday_daily_avg
    .reset_index()
    .rename(columns={'index': '요일'})
)

output_path = output_dir / '요일별_일평균_탑승건수.csv'
weekday_daily_avg.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'저장 완료: {output_path}')
weekday_daily_avg

In [ ]:
# 6. 시간대별_일평균_탑승건수.csv 생성

hour_total = (
    df_boarding_hour.groupby(['시간', '시간대_label'], as_index=False)['탑승건수']
    .sum()
    .sort_values('시간')
    .reset_index(drop=True)
)

hour_daily_avg = (
    hour_total[['시간대_label', '탑승건수']]
    .rename(columns={
        '시간대_label': '시간대',
        '탑승건수': '총탑승건수'
    })
)

hour_daily_avg['운행일수'] = total_service_days
hour_daily_avg['일평균탑승건수'] = (
    hour_daily_avg['총탑승건수'] / hour_daily_avg['운행일수']
)

hour_daily_avg = hour_daily_avg[
    ['시간대', '운행일수', '총탑승건수', '일평균탑승건수']
]

output_path = output_dir / '시간대별_일평균_탑승건수.csv'
hour_daily_avg.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'저장 완료: {output_path}')
hour_daily_avg

In [ ]:
# 7. 요일X시간대별_일평균_탑승건수.csv 생성

weekday_hour_total = (
    df_boarding_hour.pivot_table(
        index='요일',
        columns='시간대_label',
        values='탑승건수',
        aggfunc='sum',
        fill_value=0
    )
    .reindex(weekday_order)
)

weekday_hour_total = weekday_hour_total[
    [col for col in hour_columns if col in weekday_hour_total.columns]
]

weekday_hour_daily_avg = weekday_hour_total.div(
    weekday_service_days,
    axis=0
)

weekday_hour_daily_avg.index.name = '요일'

output_path = output_dir / '요일X시간대별_일평균_탑승건수.csv'
weekday_hour_daily_avg.to_csv(output_path, encoding='utf-8-sig')

print(f'저장 완료: {output_path}')
weekday_hour_daily_avg